# KNN - Stratified 5-Fold CV + Holdout Test
- Grid search on 80% train using StratifiedGroupKFold with your strat_key
- Best config selected by mean weighted F1 across 5 folds
- All 5 metrics (Accuracy, Precision, Recall, F1, AUROC) reported per fold + mean ± std
- Final evaluation on fixed 20% holdout test

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import warnings
from dotenv import load_dotenv
warnings.filterwarnings('ignore')

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score, classification_report
)
import wandb
import os
load_dotenv()

True

In [ ]:
wandb_key = os.environ.get("WANDB_API_KEY")
if not wandb_key:
    raise ValueError("Set WANDB_API_KEY environment variable")
wandb.login(key=wandb_key)

wandb.init(
    project = "commitment-mining",
    name = "ml-KNN-tf-idf-aug",
    config = {
        "model": "KNN",
    },
    tags=["KNN", "machine-learning", "tf-idf", "augmented"],
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/rupak/.netrc


## 2. Load Data

In [ ]:
TRAIN_PATH  = 'Commitment-Mining/dataset/train_80p-aug.xlsx'
TEST_PATH   = 'Commitment-Mining/dataset/test_20p.xlsx'

TEXT_COL    = 'statements (ne)'
LABEL_COL   = 'final_label'
RANDOM_SEED = 42

train_df = pd.read_excel(TRAIN_PATH)
test_df  = pd.read_excel(TEST_PATH)

for df in [train_df, test_df]:
    df[TEXT_COL]  = df[TEXT_COL].astype(str).str.strip()
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()

wandb.log({
    "train_size" : len(train_df),
    "test_size"  : len(test_df),
})

label_counts = train_df[LABEL_COL].value_counts().to_dict()
wandb.log(label_counts)

## 3. Build Stratification Key

In [15]:
# Bin sentence length into buckets
train_df['sentence_length'] = train_df[TEXT_COL].apply(
    lambda x: pd.cut(
        [len(x.split())],
        bins=[0, 5, 10, 20, 50, 999],
        labels=['xs', 's', 'm', 'l', 'xl']
    )[0]
)

# Build strat key exactly as specified
train_df['strat_key'] = (
    train_df['province'].astype(str)           + '_' +
    train_df['sentence_length'].astype(str)    + '_' +
    train_df['district/gaupalika'].astype(str) + '_' +
    train_df[LABEL_COL].astype(str)
)

# Merge rare keys (< 5 members) into 'rare' to avoid CV split issues
counts = train_df['strat_key'].value_counts()
rare   = counts[counts < 5].index
train_df['strat_key'] = train_df['strat_key'].apply(
    lambda x: 'rare' if x in rare else x
)


## 4. Prepare Features & Labels

In [16]:
le = LabelEncoder()

X_train = train_df[TEXT_COL].values
y_train = le.fit_transform(train_df[LABEL_COL].values)
groups  = train_df['strat_key'].values   # used by StratifiedGroupKFold

X_test  = test_df[TEXT_COL].values
y_test  = le.transform(test_df[LABEL_COL].values)

wandb.log({
    "num_classes": len(le.classes_),
    "classes": ", ".join(map(str, le.classes_)),
    "x_train_samples": X_train.shape[0],
    "x_train_features": X_train.shape[1] if len(X_train.shape) > 1 else 1,
    "x_test_samples": X_test.shape[0],
    "x_test_features": X_test.shape[1] if len(X_test.shape) > 1 else 1,
})

## 5. Grid Search - Find Best Config
- Uses `StratifiedGroupKFold` so your strat_key is respected
- Best config = highest mean weighted F1 across 5 validation folds
- `refit=True` → best config is refit on full 80% train automatically

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='word',
        token_pattern=r'\S+',
        sublinear_tf=True
    )),
    ('clf', KNeighborsClassifier())
])

param_grid = {
    'tfidf__ngram_range' : [(1, 1), (1, 2)],
    'tfidf__max_features': [5000, 10000, 20000],
    'clf__n_neighbors'   : [3, 5, 7, 9, 11],
    'clf__metric'        : ['cosine', 'manhattan', 'euclidean'],
    'clf__weights'       : ['uniform', 'distance']
}
# Total: 2 x 3 x 5 x 3 x 2 = 180 configs x 5 folds = 900 fits

# StratifiedGroupKFold: respects both class balance AND your strat_key groups
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

grid_search = GridSearchCV(
    estimator  = pipeline,
    param_grid = param_grid,
    cv         = sgkf,
    scoring    = 'f1_macro',   # selects best config by weighted F1
    n_jobs     = -1,
    verbose    = 1,
    refit      = True
)

# groups= is now correctly used by StratifiedGroupKFold
grid_search.fit(X_train, y_train, groups=groups)

print('\n✓ Grid search complete.') 
print(f'Best CV weighted F1 : {grid_search.best_score_:.4f}') 
print(f'Best params:') 
for k, v in grid_search.best_params_.items(): 
    print(f' {k}: {v}')

wandb.config.update({
    "best_cv_weighted_f1": grid_search.best_score_,
    **{f"best_param/{k}": v for k, v in grid_search.best_params_.items()}
})

Fitting 5 folds for each of 96 candidates, totalling 480 fits

✓ Grid search complete.
Best CV weighted F1 : 0.8565
Best params:
 clf__C: 1.0
 clf__class_weight: balanced
 clf__penalty: l2
 tfidf__max_features: 10000
 tfidf__ngram_range: (1, 1)


## 6. Per-Fold CV - All 5 Metrics with Best Config
GridSearchCV only tracked F1 during search.  
Now rerun best config across 5 folds to get Accuracy, Precision, Recall, F1, AUROC per fold.

In [ ]:
def compute_metrics(y_true, y_pred, y_proba):
    """Compute all 5 metrics. Returns a dict."""
    return {
        'accuracy'  : accuracy_score(y_true, y_pred),
        'precision' : precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall'    : recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1'        : f1_score(y_true, y_pred, average='macro', zero_division=0),
        'auroc'     : roc_auc_score(y_true, y_proba)
    }

fold_metrics = []

for fold, (tr_idx, val_idx) in enumerate(
    sgkf.split(X_train, y_train, groups=groups)
):
    X_tr,  X_val  = X_train[tr_idx],  X_train[val_idx]
    y_tr,  y_val  = y_train[tr_idx],  y_train[val_idx]

    # Fresh clone of best config — same hyperparameters, retrained from scratch
    fold_model = clone(grid_search.best_estimator_)
    fold_model.fit(X_tr, y_tr)

    y_pred  = fold_model.predict(X_val)
    y_proba = fold_model.predict_proba(X_val)[:, 1]

    m = compute_metrics(y_val, y_pred, y_proba)
    m['fold'] = fold + 1
    fold_metrics.append(m)

    print(f"Fold {fold+1} | "
          f"Acc: {m['accuracy']:.4f} | "
          f"Prec: {m['precision']:.4f} | "
          f"Rec: {m['recall']:.4f} | "
          f"F1: {m['f1']:.4f} | "
          f"AUROC: {m['auroc']:.4f}")

# Build results dataframe
fold_df = pd.DataFrame(fold_metrics).set_index('fold')
mean_row = fold_df.mean().rename('mean')
std_row  = fold_df.std().rename('std')
cv_summary = pd.concat([fold_df, mean_row.to_frame().T, std_row.to_frame().T])

print('\n=== CV Results (best config) ===') 
print(cv_summary.round(4))

wandb.log({
    "cv_results": wandb.Table(dataframe=cv_summary.round(4))
})

Fold 1 | Acc: 0.8935 | Prec: 0.8936 | Rec: 0.8935 | F1: 0.8934 | AUROC: 0.9404
Fold 2 | Acc: 0.9050 | Prec: 0.9051 | Rec: 0.9050 | F1: 0.9050 | AUROC: 0.9648
Fold 3 | Acc: 0.7521 | Prec: 0.7550 | Rec: 0.7521 | F1: 0.7482 | AUROC: 0.7610
Fold 4 | Acc: 0.8970 | Prec: 0.9017 | Rec: 0.8970 | F1: 0.8960 | AUROC: 0.9610
Fold 5 | Acc: 0.8381 | Prec: 0.8490 | Rec: 0.8381 | F1: 0.8398 | AUROC: 0.8868

=== CV Results (best config) ===
      accuracy  precision  recall      f1   auroc
1       0.8935     0.8936  0.8935  0.8934  0.9404
2       0.9050     0.9051  0.9050  0.9050  0.9648
3       0.7521     0.7550  0.7521  0.7482  0.7610
4       0.8970     0.9017  0.8970  0.8960  0.9610
5       0.8381     0.8490  0.8381  0.8398  0.8868
mean    0.8571     0.8609  0.8571  0.8565  0.9028
std     0.0644     0.0633  0.0644  0.0657  0.0851


## 7. Final Evaluation on Holdout Test Set (20%)
Best config already refit on full 80% train by GridSearchCV (`refit=True`).  
Evaluated exactly once — never used before this step.

In [19]:
y_pred_test  = grid_search.best_estimator_.predict(X_test)
y_proba_test = grid_search.best_estimator_.predict_proba(X_test)[:, 1]

test_metrics = compute_metrics(y_test, y_pred_test, y_proba_test)

print('=== HOLDOUT TEST SET RESULTS ===') 
print(f" Accuracy : {test_metrics['accuracy']:.4f}") 
print(f" Precision : {test_metrics['precision']:.4f}") 
print(f" Recall : {test_metrics['recall']:.4f}") 
print(f" F1 : {test_metrics['f1']:.4f}") 
print(f" AUROC : {test_metrics['auroc']:.4f}") 
print('\nClassification Report:') 
print(classification_report(y_test, y_pred_test, target_names=le.classes_))


wandb.log({
    "test/accuracy": test_metrics["accuracy"],
    "test/precision": test_metrics["precision"],
    "test/recall": test_metrics["recall"],
    "test/f1": test_metrics["f1"],
    "test/auroc": test_metrics["auroc"],
})

report = classification_report(
    y_test,
    y_pred_test,
    target_names=le.classes_,
    output_dict=True
)

report_df = pd.DataFrame(report).transpose().round(4)

wandb.log({
    "classification_report": wandb.Table(dataframe=report_df)
})

=== HOLDOUT TEST SET RESULTS ===
 Accuracy : 0.8477
 Precision : 0.8477
 Recall : 0.8477
 F1 : 0.8477
 AUROC : 0.9375

Classification Report:
              precision    recall  f1-score   support

           C       0.86      0.86      0.86       270
          NC       0.83      0.83      0.83       216

    accuracy                           0.85       486
   macro avg       0.85      0.85      0.85       486
weighted avg       0.85      0.85      0.85       486



## 8. Final Summary Table
CV mean ± std (validation) vs holdout test - everything in one place for the paper.

In [ ]:
metrics_order = ['accuracy', 'precision', 'recall', 'f1', 'auroc']

summary = pd.DataFrame({
    'CV Mean' : fold_df[metrics_order].mean().round(4),
    'CV Std'  : fold_df[metrics_order].std().round(4),
    'Test'    : pd.Series(test_metrics)[metrics_order].round(4)
})

print('=== PAPER TABLE — KNN (TF-IDF) ===') 
print(summary) 

print('\nBest hyperparameters:') 
for k, v in grid_search.best_params_.items(): print(f' {k}: {v}')

wandb.log({
    "paper_table": wandb.Table(dataframe=summary)
})

# Log the best hyperparameters
wandb.config.update({
    **{f"best_param/{k}": v for k, v in grid_search.best_params_.items()}
})

=== PAPER TABLE — Logistic Regression (TF-IDF) ===
           CV Mean  CV Std    Test
accuracy    0.8571  0.0644  0.8477
precision   0.8609  0.0633  0.8477
recall      0.8571  0.0644  0.8477
f1          0.8565  0.0657  0.8477
auroc       0.9028  0.0851  0.9375

Best hyperparameters:
 clf__C: 1.0
 clf__class_weight: balanced
 clf__penalty: l2
 tfidf__max_features: 10000
 tfidf__ngram_range: (1, 1)


In [21]:
wandb.finish()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


C,▁
NC,▁
num_classes,▁
test/accuracy,▁
test/auroc,▁
test/f1,▁
test/precision,▁
test/recall,▁
test_size,▁
train_size,▁
+4,...
